# Getting Started with Metaeval

This notebook introduces the core concepts and basic usage of the metaeval toolkit.

## Installation

Install metaeval using pip:

```bash
pip install metaeval
```

Or install from source:

```bash
git clone https://github.com/ryan-a-bell/dissertation.git
cd dissertation/metaeval
pip install -e .
```

In [ ]:
# Import core modules
import pandas as pd
import numpy as np

from metaeval.core.config import get_config, set_config, Config
from metaeval.benchmark.download import download_benchmark
from metaeval.bias.detection import PositionBiasAnalyzer
from metaeval.compare.analysis import FormatComparator

## Configuration

Metaeval uses a centralized configuration system. You can view and modify settings:

In [ ]:
# View current configuration
config = get_config()

print("Judge Settings:")
print(f"  Temperature: {config.judge.temperature}")
print(f"  Max tokens: {config.judge.max_tokens}")
print(f"  Default provider: {config.judge.default_provider}")

print("\nStats Settings:")
print(f"  Alpha: {config.stats.alpha}")
print(f"  Bootstrap iterations: {config.stats.bootstrap_iterations}")

In [ ]:
# Customize configuration
custom_config = Config()
custom_config.stats.alpha = 0.01  # More strict significance level
custom_config.judge.temperature = 0.1  # Slightly less deterministic

set_config(custom_config)
print("Configuration updated!")

## Downloading Benchmark Data

Download benchmark datasets from Hugging Face:

In [ ]:
# Download SysEngBench (or use your own data)
# df = download_benchmark("ryan-a-bell/SysEngBench")

# For this example, we'll create synthetic data
np.random.seed(42)

def create_sample_data(n_questions=100, n_models=3):
    """Create sample MCQ benchmark data."""
    positions = ["A", "B", "C", "D"]
    data = []
    
    for model_idx in range(n_models):
        model = f"model_{model_idx}"
        for q_id in range(n_questions):
            for pos in positions:
                # Add some position bias for demonstration
                bias = 0.15 if pos == "A" else 0.0
                correct = np.random.random() < (0.6 + bias)
                data.append({
                    "question_id": q_id,
                    "model": model,
                    "variant": pos,
                    "is_correct": int(correct),
                })
    
    return pd.DataFrame(data)

df = create_sample_data()
print(f"Created dataset with {len(df)} rows")
df.head(10)

## Position Bias Analysis

Detect if models show bias toward certain answer positions:

In [ ]:
# Create analyzer
analyzer = PositionBiasAnalyzer(df)

# Analyze a single model
report = analyzer.analyze("model_0")

print(f"Model: {report.model}")
print(f"\nAccuracy by Position:")
for pos, acc in report.accuracy_by_position.items():
    print(f"  {pos}: {acc:.3f}")

print(f"\nSignificant Bias Detected: {report.has_significant_bias}")
print(f"Chi-square p-value: {report.test_results['chi_square']['p_value']:.4f}")
print(f"Cramér's V: {report.effect_sizes['cramers_v']['value']:.4f}")

In [ ]:
# Analyze all models
all_reports = analyzer.analyze_all_models()

print("Summary of Position Bias Across Models:")
print("="*50)
for model, report in all_reports.items():
    bias_status = "YES" if report.has_significant_bias else "No"
    print(f"{model}: Bias={bias_status}, Cramér's V={report.effect_sizes['cramers_v']['value']:.4f}")

## Next Steps

- Check out `02_position_bias_analysis.ipynb` for detailed bias analysis
- See `03_mcq_to_osq_conversion.ipynb` for converting questions
- Explore `04_llm_as_judge.ipynb` for judging responses